In [ ]:
# === gridsearch_tuning.ipynb — Setup ===
import glob
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# Load the FROZEN dataset
matches = glob.glob('**/model_dataset.csv', recursive=True) + \
          glob.glob('../**/model_dataset.csv', recursive=True)
if not matches:
    raise FileNotFoundError("model_dataset.csv not found — run baseline_comparison first")
print("Loading:", matches[0])
model_df = pd.read_csv(matches[0])

pred_cols = ['dist_roads','dist_parks','dist_coca','dist_mosaic',
             'temp_C','vpd_kPa','ndvi','wind_ms','oni']

# Spatial blocks
BLOCK = 0.25
model_df['block'] = (model_df['lon']//BLOCK).astype(int).astype(str) + '_' + \
                    (model_df['lat']//BLOCK).astype(int).astype(str)

print("Dataset:", model_df.shape, "| blocks:", model_df['block'].nunique())

Loading: ..\log_regression\datasets\model_dataset.csv
Dataset: (6231, 14) | blocks: 120


In [19]:
# === Tuned models — full evaluation with the SAME output format as before ===
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.base import clone

# The tuned models (best configs found by GridSearch)
best_lr = lr_search.best_estimator_      # Pipeline: StandardScaler + LogisticRegression(C=0.01, L1)
best_rf = rf_search.best_estimator_      # RandomForest(n_est=500, sqrt, leaf=5, depth=None)

print("Tuned LR:", lr_search.best_params_)
print("Tuned RF:", rf_search.best_params_)

Tuned LR: {'clf__C': 0.01, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}
Tuned RF: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 500}


In [20]:
def evaluate_model(name, estimator, model_df, pred_cols, n_splits=10, block=0.25):
    """Spatial block CV (with uncertainty) + temporal split — identical format for any model."""
    X = model_df[pred_cols].values
    y = model_df['burned'].astype(int).values

    # Spatial blocks
    blocks = (model_df['lon']//block).astype(int).astype(str) + '_' + \
             (model_df['lat']//block).astype(int).astype(str)
    groups = blocks.values

    # ---------- SPATIAL BLOCK CV ----------
    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups), 1):
        m = clone(estimator).fit(X[tr], y[tr])          # fresh copy, refit per fold
        prob = m.predict_proba(X[te])[:, 1]
        pred = m.predict(X[te])
        auc   = roc_auc_score(y[te], prob)
        prauc = average_precision_score(y[te], prob)
        f1    = f1_score(y[te], pred)
        rows.append((auc, prauc, f1))
        print(f"  Fold {fold:2d}: AUC={auc:.3f}  PR-AUC={prauc:.3f}  F1={f1:.3f}")

    r = np.array(rows)
    print(f"\nSPATIAL BLOCK CV — {name}")
    print(f"  AUC-ROC : {r[:,0].mean():.3f} ± {r[:,0].std():.3f}")
    print(f"  PR-AUC  : {r[:,1].mean():.3f} ± {r[:,1].std():.3f}")
    print(f"  F1      : {r[:,2].mean():.3f} ± {r[:,2].std():.3f}")

    # ---------- TEMPORAL SPLIT ----------
    tr = (model_df['year'] <= 2019).values
    te = (model_df['year'] >= 2020).values
    m = clone(estimator).fit(X[tr], y[tr])
    prob = m.predict_proba(X[te])[:, 1]
    pred = m.predict(X[te])

    print(f"\nTEMPORAL SPLIT — {name}")
    print(f"  train ≤2019: {tr.sum()} rows ({int(y[tr].sum())} events) | "
          f"test ≥2020: {te.sum()} rows ({int(y[te].sum())} events)")
    print(f"  AUC-ROC : {roc_auc_score(y[te], prob):.3f}")
    print(f"  PR-AUC  : {average_precision_score(y[te], prob):.3f}")
    print(f"  F1      : {f1_score(y[te], pred):.3f}")

    return {'spatial': r.mean(axis=0), 'spatial_std': r.std(axis=0),
            'temporal': (roc_auc_score(y[te], prob),
                         average_precision_score(y[te], prob),
                         f1_score(y[te], pred))}

In [25]:
# evaluate the tuned models

res_lr = evaluate_model("Logistic Regression (tuned)", best_lr, model_df, pred_cols)


res_rf = evaluate_model("Random Forest (tuned)", best_rf, model_df, pred_cols)

c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ra

  Fold  1: AUC=0.899  PR-AUC=0.839  F1=0.808
  Fold  2: AUC=0.671  PR-AUC=0.502  F1=0.580
  Fold  3: AUC=0.848  PR-AUC=0.760  F1=0.668
  Fold  4: AUC=0.855  PR-AUC=0.790  F1=0.705
  Fold  5: AUC=0.878  PR-AUC=0.706  F1=0.700
  Fold  6: AUC=0.841  PR-AUC=0.713  F1=0.658
  Fold  7: AUC=0.831  PR-AUC=0.667  F1=0.612
  Fold  8: AUC=0.865  PR-AUC=0.797  F1=0.718
  Fold  9: AUC=0.769  PR-AUC=0.540  F1=0.604
  Fold 10: AUC=0.873  PR-AUC=0.729  F1=0.711

SPATIAL BLOCK CV — Logistic Regression (tuned)
  AUC-ROC : 0.833 ± 0.063
  PR-AUC  : 0.704 ± 0.103
  F1      : 0.676 ± 0.064


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



TEMPORAL SPLIT — Logistic Regression (tuned)
  train ≤2019: 5153 rows (1828 events) | test ≥2020: 1078 rows (249 events)
  AUC-ROC : 0.807
  PR-AUC  : 0.617
  F1      : 0.529
  Fold  1: AUC=0.919  PR-AUC=0.881  F1=0.824
  Fold  2: AUC=0.777  PR-AUC=0.646  F1=0.652
  Fold  3: AUC=0.862  PR-AUC=0.777  F1=0.719
  Fold  4: AUC=0.884  PR-AUC=0.817  F1=0.760
  Fold  5: AUC=0.912  PR-AUC=0.773  F1=0.683
  Fold  6: AUC=0.890  PR-AUC=0.808  F1=0.730
  Fold  7: AUC=0.896  PR-AUC=0.780  F1=0.706
  Fold  8: AUC=0.886  PR-AUC=0.817  F1=0.744
  Fold  9: AUC=0.839  PR-AUC=0.693  F1=0.645
  Fold 10: AUC=0.916  PR-AUC=0.848  F1=0.766

SPATIAL BLOCK CV — Random Forest (tuned)
  AUC-ROC : 0.878 ± 0.041
  PR-AUC  : 0.784 ± 0.066
  F1      : 0.723 ± 0.052

TEMPORAL SPLIT — Random Forest (tuned)
  train ≤2019: 5153 rows (1828 events) | test ≥2020: 1078 rows (249 events)
  AUC-ROC : 0.817
  PR-AUC  : 0.564
  F1      : 0.548


In [24]:
# --- LR coefficients (from the tuned pipeline) ---
coefs = best_lr.named_steps['clf'].coef_[0]
coef_df = pd.DataFrame({'predictor': pred_cols, 'coefficient': coefs}) \
            .sort_values('coefficient', key=abs, ascending=False)
print("\nStandardized coefficients — TUNED LR (L1, C=0.01):")
print(coef_df.round(3).to_string(index=False))
zeroed = coef_df[coef_df['coefficient'] == 0]['predictor'].tolist()
print(f"Eliminated by L1: {zeroed if zeroed else 'none'}")

# --- RF impurity importance ---
X_all = model_df[pred_cols].values
y_all = model_df['burned'].astype(int).values
rf_fit = clone(best_rf).fit(X_all, y_all)
imp = pd.DataFrame({'predictor': pred_cols, 'importance': rf_fit.feature_importances_}) \
        .sort_values('importance', ascending=False)
print("\nRF impurity importance — TUNED:")
print(imp.round(3).to_string(index=False))


Standardized coefficients — TUNED LR (L1, C=0.01):
  predictor  coefficient
       ndvi       -0.667
    vpd_kPa        0.619
    wind_ms        0.540
 dist_parks        0.277
 dist_roads       -0.181
     temp_C        0.085
  dist_coca        0.000
dist_mosaic        0.000
        oni        0.000
Eliminated by L1: ['dist_coca', 'dist_mosaic', 'oni']

RF impurity importance — TUNED:
  predictor  importance
       ndvi       0.224
    wind_ms       0.201
    vpd_kPa       0.159
     temp_C       0.109
 dist_parks       0.083
        oni       0.075
 dist_roads       0.068
  dist_coca       0.046
dist_mosaic       0.035


In [26]:
import pandas as pd

# Results you already computed (spatial CV: mean ± std | temporal: single value)
results = {
    'LR — before tuning': {
        'auc_sp': (0.836, 0.029), 'prauc_sp': (0.692, 0.101), 'f1_sp': (0.666, 0.059),
        'auc_tp': 0.808, 'prauc_tp': 0.620, 'f1_tp': 0.521},
    'LR — after tuning': {
        'auc_sp': (0.833, 0.063), 'prauc_sp': (0.704, 0.103), 'f1_sp': (0.676, 0.064),
        'auc_tp': 0.807, 'prauc_tp': 0.617, 'f1_tp': 0.529},
    'RF — before tuning': {
        'auc_sp': (0.875, 0.022), 'prauc_sp': (0.769, 0.067), 'f1_sp': (0.709, 0.060),
        'auc_tp': 0.817, 'prauc_tp': 0.563, 'f1_tp': 0.549},
    'RF — after tuning': {
        'auc_sp': (0.878, 0.041), 'prauc_sp': (0.784, 0.066), 'f1_sp': (0.723, 0.052),
        'auc_tp': 0.817, 'prauc_tp': 0.564, 'f1_tp': 0.548},
}

def fmt(v):
    return f"{v[0]:.3f}±{v[1]:.3f}" if isinstance(v, tuple) else f"{v:.3f}"

print("="*104)
print("SPATIAL BLOCK CV (validation, mean ± std across 10 folds)")
print("="*104)
print(f"{'Model':22s} {'AUC-ROC':>16s} {'PR-AUC':>16s} {'F1':>16s}")
for k, r in results.items():
    print(f"{k:22s} {fmt(r['auc_sp']):>16s} {fmt(r['prauc_sp']):>16s} {fmt(r['f1_sp']):>16s}")

print("\n" + "="*104)
print("TEMPORAL HOLD-OUT (validation on unseen years ≥2020 — no leakage)")
print("="*104)
print(f"{'Model':22s} {'AUC-ROC':>10s} {'PR-AUC':>10s} {'F1':>10s}")
for k, r in results.items():
    print(f"{k:22s} {fmt(r['auc_tp']):>10s} {fmt(r['prauc_tp']):>10s} {fmt(r['f1_tp']):>10s}")

# Explicit deltas from tuning
print("\n" + "="*104)
print("EFFECT OF TUNING (Δ PR-AUC)")
print("="*104)
for model in ['LR', 'RF']:
    d_sp = results[f'{model} — after tuning']['prauc_sp'][0] - results[f'{model} — before tuning']['prauc_sp'][0]
    d_tp = results[f'{model} — after tuning']['prauc_tp']    - results[f'{model} — before tuning']['prauc_tp']
    print(f"  {model}: spatial {d_sp:+.3f}  |  temporal {d_tp:+.3f}")

SPATIAL BLOCK CV (validation, mean ± std across 10 folds)
Model                           AUC-ROC           PR-AUC               F1
LR — before tuning          0.836±0.029      0.692±0.101      0.666±0.059
LR — after tuning           0.833±0.063      0.704±0.103      0.676±0.064
RF — before tuning          0.875±0.022      0.769±0.067      0.709±0.060
RF — after tuning           0.878±0.041      0.784±0.066      0.723±0.052

TEMPORAL HOLD-OUT (validation on unseen years ≥2020 — no leakage)
Model                     AUC-ROC     PR-AUC         F1
LR — before tuning          0.808      0.620      0.521
LR — after tuning           0.807      0.617      0.529
RF — before tuning          0.817      0.563      0.549
RF — after tuning           0.817      0.564      0.548

EFFECT OF TUNING (Δ PR-AUC)
  LR: spatial +0.012  |  temporal -0.003
  RF: spatial +0.015  |  temporal +0.001
